# VideoSearch · Colab Control Room

Terminal komutu yazmadan GPU pipeline, görsel query sonuçları, accuracy ve rapor paketi.

Önce **Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU** seçin; ardından **Tümünü çalıştır** deyin.

In [ ]:
#@title 1 · Hazır POC paketini yükle
import pathlib, shutil, zipfile
from IPython.display import HTML, display
from google.colab import files

display(HTML("<h3>video-search-poc-colab.zip paketini seç</h3><p>Kod paketi küçüktür; 7.5 GB VisDrone verisi buna dahil değildir.</p>"))
uploaded = files.upload()
archives = [pathlib.Path(name) for name in uploaded if name.lower().endswith('.zip')]
if not archives:
    raise RuntimeError('ZIP paketi seçilmedi.')
extract_root = pathlib.Path('/content/video_search_bundle')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(archives[0]) as bundle:
    for info in bundle.infolist():
        member = pathlib.PurePosixPath(info.filename)
        if member.is_absolute() or '..' in member.parts:
            raise RuntimeError(f'Güvensiz paket yolu: {info.filename}')
    bundle.extractall(extract_root)
matches = list(extract_root.rglob('notebooks/colab_dashboard.py'))
if len(matches) != 1:
    raise RuntimeError(f'POC kökü bulunamadı: {len(matches)} eşleşme')
REPO_ROOT = matches[0].parents[1]
display(HTML(f"<b style='color:#067647'>✓ Paket hazır:</b> {REPO_ROOT}"))

In [ ]:
#@title 2 · Bağımlılıkları sessizce kur
import subprocess, sys
from IPython.display import HTML, display
display(HTML("<b>Model ve dashboard bağımlılıkları kuruluyor…</b>"))
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'notebooks/requirements-colab.txt')],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
if result.returncode:
    raise RuntimeError(result.stdout[-5000:])
display(HTML("<b style='color:#067647'>✓ Kurulum tamamlandı.</b>"))

In [ ]:
#@title 3 · Control Room'u aç
import os, sys
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
from notebooks.colab_dashboard import create_gradio_app
APP = create_gradio_app(REPO_ROOT)
APP.launch(share=True, inline=True, debug=False, show_error=True)